# Validadores *before*

Na maioria das vezes rodamos nossos validadores **depois** que o Pydantic já processou os dados — os chamados **validadores after**.

A grande vantagem é que o validador tem garantia de que o valor recebido tem o tipo correto e já foi validado conforme especificamos no modelo.

Às vezes, porém, queremos interceptar os dados **antes** de o Pydantic rodar seus validadores — são os **validadores before**.

O motivo é precisar customizar a desserialização antes da validação do próprio Pydantic.

Por exemplo, num modelo com campo datetime, a coerção de string para datetime do Pydantic só funciona se a string estiver num formato ISO específico.

In [1]:
from datetime import datetime

from pydantic import BaseModel, field_validator, ValidationError

In [2]:
class Model(BaseModel):
    dt: datetime

In [3]:
print(f'Model(dt="2020-01-01T12:00:00"): {Model(dt="2020-01-01T12:00:00")!r}')

Model(dt="2020-01-01T12:00:00"): Model(dt=datetime.datetime(2020, 1, 1, 12, 0))


In [4]:
print(f'Model(dt="2020-01-01T12:00:00Z"): {Model(dt="2020-01-01T12:00:00Z")!r}')

Model(dt="2020-01-01T12:00:00Z"): Model(dt=datetime.datetime(2020, 1, 1, 12, 0, tzinfo=TzInfo(0)))


Mas nenhum destes funciona:

In [5]:
try:
    Model(dt="2020/1/1 3:00pm")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
dt
  Input should be a valid datetime or date, invalid date separator, expected `-` [type=datetime_from_date_parsing, input_value='2020/1/1 3:00pm', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/datetime_from_date_parsing


In [6]:
try:
    Model(dt="Jan 1, 2020 3:00pm")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
dt
  Input should be a valid datetime or date, invalid character in year [type=datetime_from_date_parsing, input_value='Jan 1, 2020 3:00pm', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/datetime_from_date_parsing


Podemos escrever nosso próprio validador para desserializar essas strings, mas não pode ser **after**: a validação do Pydantic falharia
antes de o nosso validador rodar.

Então usamos um validador **before**.

Primeiro, código Python capaz de transformar essas várias strings de data em objetos datetime.

Vamos usar a biblioteca de terceiros `python-dateutil`.

Instale no ambiente virtual; a documentação está [aqui](https://github.com/dateutil/dateutil).

Em particular, usaremos o *parser* dessa biblioteca.

In [7]:
from dateutil.parser import parse

In [8]:
print(f'parse("2020/1/1 3pm"): {parse("2020/1/1 3pm")!r}')

parse("2020/1/1 3pm"): datetime.datetime(2020, 1, 1, 15, 0)


In [9]:
print(f'parse("Jan 1, 2020 3pm"): {parse("Jan 1, 2020 3pm")!r}')

parse("Jan 1, 2020 3pm"): datetime.datetime(2020, 1, 1, 15, 0)


Mas a função só funciona com string, não com objeto datetime:

In [10]:
try:
    parse(datetime(2020, 1, 1, 15, 0, 0))
except TypeError as ex:
    print(f"{type(ex).__name__}: {ex}")

TypeError: Parser must be a string or character stream, not datetime


Num validador before, recebemos o valor **cru**, que pode ser string, datetime ou qualquer outro objeto.

Precisamos tratar isso no validador.

Um validador **before** é definido do mesmo jeito que um **after**; basta um argumento indicando o modo (o padrão é after).

Tecnicamente, não precisamos devolver o tipo final do modelo, já que os validadores do Pydantic rodam depois — mas poderíamos. Aqui,
tentamos interpretar o valor se for string; caso contrário, repassamos o valor como veio e deixamos o Pydantic tratar entradas não-string.

In [11]:
from typing import Any


class Model(BaseModel):
    dt: datetime

    @field_validator("dt", mode="before")
    @classmethod
    def parse_datetime(cls, value: Any):
        if isinstance(value, str):
            print("parsing string")
            try:
                return parse(value)
            except Exception as ex:
                raise ValueError(str(ex))
        print("pass through...")
        return value

In [12]:
print(f'Model(dt="2020/1/1 3pm"): {Model(dt="2020/1/1 3pm")!r}')

parsing string
Model(dt="2020/1/1 3pm"): Model(dt=datetime.datetime(2020, 1, 1, 15, 0))


Qualquer outro tipo passa pelo nosso validador como está e é tratado pelos validadores do Pydantic:

In [13]:
print(f"Model(dt=datetime(2020, 1, 1)): {Model(dt=datetime(2020, 1, 1))!r}")

pass through...
Model(dt=datetime(2020, 1, 1)): Model(dt=datetime.datetime(2020, 1, 1, 0, 0))


In [14]:
try:
    Model(dt=[1, 2, 3])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

pass through...
ValidationError: 1 validation error for Model
dt
  Input should be a valid datetime [type=datetime_type, input_value=[1, 2, 3], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/datetime_type


Então lembre: com validadores **before**, o validador recebe o valor **cru**, como veio nos dados (ou do validador before anterior, se houver).

Podemos fazer qualquer validação ou transformação e devolver um valor. Ao devolver, roda o próximo validador before; se não houver
mais, o Pydantic assume o restante da validação, seguido dos validadores after (se houver).

> 💡 **Complemento:** repare no `raise ValueError(str(ex))` dentro do `except Exception` — é o padrão correto: transforma qualquer falha do
> parser num erro de validação (lembre que outras exceções "vazariam" sem virar `ValidationError`).

In [15]:
try:
    Model(dt="não é data")
except ValidationError as ex:
    print(f'ex.errors()[0]["type"]: {ex.errors()[0]["type"]}')
    print(f'ex.errors()[0]["msg"]: {ex.errors()[0]["msg"]}')

parsing string
ex.errors()[0]["type"]: value_error
ex.errors()[0]["msg"]: Value error, Unknown string format: não é data


Por fim, a ordem de execução quando há vários validadores before.

Como vimos nos slides, o Pydantic executa os validadores before de **baixo para cima** na ordem de definição.

Vejamos rapidamente:

In [16]:
class Model(BaseModel):
    number: int

    @field_validator("number", mode="before")
    @classmethod
    def validator_1(cls, value):
        print("running validator_1")
        return value

    @field_validator("number", mode="before")
    @classmethod
    def validator_2(cls, value):
        print("running validator_2")
        return value

    @field_validator("number", mode="before")
    @classmethod
    def validator_3(cls, value):
        print("running validator_3")
        return value

In [17]:
print(f"Model(number=1): {Model(number=1)!r}")

running validator_3
running validator_2
running validator_1
Model(number=1): Model(number=1)


A ordem é de baixo para cima.